**Question:** How much of the GRU + XGBoost blend (notebook 16) survives when the XGBoost partner is a *pair-only* variant, i.e. B' without the 8 match-context columns, so that it can be computed for any pair of players without a concrete match (league, season size and rank-in-league are unknown for a hypothetical pair)?

**Assumptions:** Same test matches and folds as notebooks 14-16 (walk-forward folds 1-4, 14,428 matches pooled, 28,856 rows). The pair-only feature set is the same 29 columns as TabPFN `D-Bprime-par` (B' = 37 columns minus the 8 match-context columns, `tabpfn_data.CONTEXT_COLS`). Blend weights are fixed in advance (0.5 / 0.5). GRU seeds are used raw (no isotonic step); XGBoost is isotonic-calibrated on the calibration slice of its fold.

**Method:** XGBoost is re-trained walk-forward with `scripts/run_walkforward_xgb_bprime.py --variant Bprime_par`: identical hyper-parameters, chronological train / early-stopping / calibration split and seed as the B' run. Paired bootstrap by match (2,000 resamples, seed 42, the same indices for every comparison), pooled over 4 folds; Brier, accuracy (both perspectives, a tie counts half a point) and AUC; symmetry check before and after `p_sym = (p_AB + 1 - p_BA) / 2`.

**Acceptance rule (fixed before looking at the results):** the pair-only blend, symmetrised, is accepted if its paired dBrier versus the raw 5-seed GRU ensemble has a 95% CI entirely below 0 **and** it is better in at least 3 of 4 folds. If the CI is below 0 only versus the single raw GRU (C3 raw), the result is "borderline, decision needed"; otherwise "not accepted".

# 17 -- GRU + pair-only XGBoost blend (B' without match context)

**Why a pair-only variant.** The 8 match-context columns (`liga_koeficjent`, `liga_tip`, `broj_liga_u_sezoni`, `broj_igraca_ukupno`, `broj_igraca_uligi`, `relative_razina_by_liga_count`, `relative_razina_by_player_count`, `relative_razina_pct`) describe the league and season of one concrete match. For an arbitrary pair of players such a match does not exist, so a model that needs them cannot be evaluated on that pair. Dropping them leaves 29 columns that are all functions of the two players and their shared history (ELO difference, form, rest, trajectory differences, head-to-head, debut features, age). This notebook measures what the blend loses by that restriction.

**Statistics.** Paired bootstrap by match (both perspective rows together). Negative dBrier = first model better. No predictions or weights are stored in this notebook.

## Step 0 -- Imports and data

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, '../scripts')
from exp_d_analysis import (load_folds, add_d35, add_xgb, add_blends_par, pool_folds, brier_ci,
                            paired_dbrier, boot_idx, summary_table, pair_dev, symmetrize)

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', None)

WORKDIR = os.environ.get('GDSI_ML_WORKDIR', '..')
# XGBoost predictions are produced once by the training script (skipped if already present)
for variant, stem in (('Bprime', 'xgb_bprime'), ('Bprime_par', 'xgb_bprimepar')):
    if not (Path(WORKDIR) / 'exp_xgb' / f'test_predictions_{stem}_fold4.parquet').exists():
        subprocess.run([sys.executable, '../scripts/run_walkforward_xgb_bprime.py', WORKDIR, '--variant', variant], check=True)

B = add_xgb(add_xgb(add_d35(load_folds(WORKDIR), WORKDIR), WORKDIR, 'Bprime'), WORKDIR, 'Bprime_par')
LAB = {'p_ELO': 'ELO', 'p_C3': 'C3 (calibrated)', 'p_C3raw': 'C3 raw', 'p_C3ens_raw': 'C3ens raw (5 seeds)'}
LAB.update(add_blends_par(B))
pool = pool_folds(B)
yp = pool.Label.values
boots = boot_idx(pool.SusretId.values)
folds = sorted(B)
brier = lambda d, c: np.mean((d[c] - d.Label) ** 2)
print('folds:', folds, '| pooled matches:', pool.SusretId.nunique(), '| rows:', len(pool))

## Step 1 -- Pair-only XGBoost versus XGBoost B'

Same hyper-parameters, same folds. The difference is only the 8 dropped context columns. Negative dBrier = first model better.

In [ ]:
print('Brier by fold')
for c in ('p_ELO', 'p_xgb', 'p_xgbpar', 'p_C3raw', 'p_C3ens_raw'):
    print(f'  {LAB[c]:24s}', [round(float(brier(B[k], c)), 5) for k in folds])
print()
for a, b_ in (('p_xgbpar', 'p_xgb'), ('p_xgbpar', 'p_C3raw'), ('p_xgbpar', 'p_ELO')):
    d, lo, hi = paired_dbrier(pool[a].values, pool[b_].values, yp, boots)
    w = sum(brier(B[k], a) < brier(B[k], b_) for k in folds)
    print(f'{LAB[a]:18s} vs {LAB[b_]:20s} dBrier {d:+.5f} [{lo:+.5f}, {hi:+.5f}]   folds better: {w}/4')

## Step 2 -- Blends with fixed 0.5 / 0.5 weights

The `symmetrised` rows use `p_sym = (p_AB + 1 - p_BA) / 2`, applied after blending. The rows with XGBoost B' are the reference from notebook 16.

In [ ]:
ORDER = ['p_ELO', 'p_C3raw', 'p_C3ens_raw', 'p_xgb', 'p_xgbpar', 'b_C3ensraw_xgb', 'b_C3ensraw_xgbpar', 'b_C3raw_xgbpar',
         'p_xgb_sym', 'p_xgbpar_sym', 'b_C3ensraw_xgb_sym', 'b_C3ensraw_xgbpar_sym', 'b_C3raw_xgbpar_sym']
AC = summary_table(pool, ORDER, LAB, boots)
print(AC.drop(columns=['ties']).round(5).to_string(index=False))
print()
print('| Model | Brier [95% CI] | % Brier vs ELO | Accuracy [95% CI] | AUC |')
print('|---|---|---|---|---|')
for _, r in AC.iterrows():
    print(f"| {r['Model']} | {r['Brier']:.5f} [{r['Brier lo']:.5f}, {r['Brier hi']:.5f}] | {r['% Brier vs ELO']:.1f}% | "
          f"{r['Acc %']:.2f}% [{r['Acc lo']:.2f}, {r['Acc hi']:.2f}] | {r['AUC']:.4f} |")

## Step 3 -- Paired dBrier of the blends (acceptance rule)

References: the raw 5-seed GRU ensemble (C3ens raw), the single raw GRU (C3 raw) and, for orientation, the blend with XGBoost B'. 'Folds better' counts folds (out of 4) with a lower Brier than the reference.

In [ ]:
CAND = ['b_C3ensraw_xgbpar', 'b_C3ensraw_xgbpar_sym', 'b_C3raw_xgbpar', 'b_C3raw_xgbpar_sym', 'b_C3ensraw_xgb', 'b_C3ensraw_xgb_sym']
REFS = ['p_C3ens_raw', 'p_C3raw', 'b_C3ensraw_xgb']
rows = []
for c in CAND:
    for ref in REFS:
        if c == ref:
            continue
        d, lo, hi = paired_dbrier(pool[c].values, pool[ref].values, yp, boots)
        rows.append({'model': LAB[c], 'reference': LAB[ref], 'dBrier': d, 'CI lo': lo, 'CI hi': hi,
                     'CI < 0': hi < 0, 'folds better': sum(brier(B[k], c) < brier(B[k], ref) for k in folds)})
print(pd.DataFrame(rows).round(5).to_string(index=False))

In [ ]:
def verdict(col):
    d1, lo1, hi1 = paired_dbrier(pool[col].values, pool['p_C3ens_raw'].values, yp, boots)
    w = sum(brier(B[k], col) < brier(B[k], 'p_C3ens_raw') for k in folds)
    d2, lo2, hi2 = paired_dbrier(pool[col].values, pool['p_C3raw'].values, yp, boots)
    if hi1 < 0 and w >= 3:
        out = 'ACCEPTED'
    elif hi2 < 0:
        out = 'BORDERLINE (CI below 0 only versus C3 raw): decision needed'
    else:
        out = 'NOT ACCEPTED'
    print(f'{LAB[col]}')
    print(f'  vs C3ens raw: dBrier {d1:+.5f} [{lo1:+.5f}, {hi1:+.5f}], folds better {w}/4')
    print(f'  vs C3 raw:    dBrier {d2:+.5f} [{lo2:+.5f}, {hi2:+.5f}]')
    print(f'  -> {out}')
    print()

verdict('b_C3ensraw_xgbpar_sym')   # the symmetrised variant is the one the rule is stated for
verdict('b_C3ensraw_xgbpar')       # unsymmetrised, for reference

## Step 4 -- Brier by fold

In [ ]:
cols = ['p_ELO', 'p_C3raw', 'p_C3ens_raw', 'p_xgb', 'p_xgbpar', 'b_C3ensraw_xgb', 'b_C3ensraw_xgbpar', 'b_C3ensraw_xgbpar_sym']
FT = pd.DataFrame([{'fold': k, **{LAB[c]: brier(B[k], c) for c in cols}} for k in folds])
print(FT.round(5).to_string(index=False))

## Step 5 -- Symmetry check

A well-formed win probability satisfies `p_AB + p_BA = 1`. Below, `|p_AB + p_BA - 1|` per match before symmetrisation, and Brier before and after `p_sym = (p_AB + 1 - p_BA) / 2`. After symmetrisation the deviation is zero by construction.

In [ ]:
rows = []
for c in ['p_C3raw', 'p_C3ens_raw', 'p_xgb', 'p_xgbpar', 'b_C3ensraw_xgb', 'b_C3ensraw_xgbpar']:
    dev = pair_dev(pool, c)
    ps = symmetrize(pool, c)
    b0, b1 = np.mean((pool[c].values - yp) ** 2), np.mean((ps - yp) ** 2)
    dev_after = pair_dev(pool.assign(_s=ps), '_s')
    rows.append({'model': LAB[c], 'mean |dev|': dev.mean(), 'max |dev|': dev.max(), 'share > 0.01': (dev > 0.01).mean(),
                 'max |dev| after': dev_after.max(), 'Brier': b0, 'Brier sym': b1, 'd Brier': b1 - b0})
print(pd.DataFrame(rows).round(5).to_string(index=False))

## Conclusion

The acceptance rule from the introduction is evaluated in Step 3 on the symmetrised pair-only blend; the line printed by `verdict` is the outcome. Step 1 quantifies how much the XGBoost partner itself loses without the match-context columns, Steps 2 and 4 show the same for the blends, and Step 5 the symmetry before and after the correction.

**Limitations.**
- Weights are fixed 0.5 / 0.5, not tuned.
- C3ens is a 5-seed average whereas XGBoost is single-seed.
- Test matches are shared across models; only paired CIs are used to compare.
- The pair-only variant is evaluated on real matches, where the dropped context does exist; its value for pairs that never met in a league match is outside this test set.